In [73]:
import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import roc_auc_score
import timm
import numpy as np

In [74]:
# --- Dataset Class ---
class CheXpertDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        # Strip "CheXpert-v1.0/" prefix from image paths if present
        self.data['Path'] = self.data['Path'].apply(lambda x: x.replace("CheXpert-v1.0/", ""))
        # Targets: assuming label columns start from 5th index onward (adjust if different)
        self.labels = self.data.iloc[:, 5:].values.astype(np.float32)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.data.iloc[idx, 1])  # Assuming Path column at index 1
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(self.labels[idx])
        return image, label


In [75]:
# --- Transforms ---
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

In [76]:
# --- Setup ---
data_root = '/scratch/smanika3/chexpert/full_uncompressed/train'
train_csv = '/scratch/smanika3/chexpert/full_uncompressed/train_cheXbert.csv'  # Adjust CSV path if needed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create datasets and loaders
full_data = pd.read_csv(train_csv)
# Simple split by patient for train-val split example (adjust for proper split)
train_df = full_data.sample(frac=0.8, random_state=42)
val_df = full_data.drop(train_df.index)

train_df.to_csv('train_split.csv', index=False)
val_df.to_csv('val_split.csv', index=False)

train_dataset = CheXpertDataset('train_split.csv', data_root, transform=train_transforms)
val_dataset = CheXpertDataset('val_split.csv', data_root, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

#

In [77]:
# --- Model ---
model = timm.create_model('convnext_base', pretrained=True, num_classes=train_dataset.labels.shape[1])
model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [78]:
# --- Training and Validation Functions ---
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    preds_all = []
    labels_all = []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds_all.append(torch.sigmoid(outputs).detach().cpu().numpy())
        labels_all.append(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    preds_all = np.vstack(preds_all)
    labels_all = np.vstack(labels_all)
    try:
        auc = roc_auc_score(labels_all, preds_all, average='macro')
    except ValueError:
        auc = float('nan')  # if only one class in labels

    return epoch_loss, auc

def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    preds_all = []
    labels_all = []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds_all.append(torch.sigmoid(outputs).cpu().numpy())
            labels_all.append(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    preds_all = np.vstack(preds_all)
    labels_all = np.vstack(labels_all)
    try:
        auc = roc_auc_score(labels_all, preds_all, average='macro')
    except ValueError:
        auc = float('nan')

    return epoch_loss, auc

In [79]:
# --- Main Training Loop ---
num_epochs = 1
train_losses, val_losses = [], []
train_aucs, val_aucs = [], []

for epoch in range(num_epochs):
    train_loss, train_auc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_auc = validate_epoch(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_aucs.append(train_auc)
    val_aucs.append(val_auc)

    print(f'Epoch {epoch+1}/{num_epochs}')
    print(f'Train Loss: {train_loss:.4f}, Train AUC: {train_auc:.4f}')
    print(f'Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}')


FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/pchalla7/.local/lib/python3.12/site-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/pchalla7/.local/lib/python3.12/site-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_37879/3224493560.py", line 17, in __getitem__
    image = Image.open(img_path).convert('RGB')
            ^^^^^^^^^^^^^^^^^^^^
  File "/packages/apps/jupyter/2025-03-24/lib/python3.12/site-packages/PIL/Image.py", line 3465, in open
    fp = builtins.open(filename, "rb")
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/scratch/smanika3/chexpert/full_uncompressed/train/Female'


In [ ]:
# --- Save metrics for plotting ---
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.legend()
plt.title('Loss per Epoch')

plt.subplot(1,2,2)
plt.plot(train_aucs, label='Train AUC')
plt.plot(val_aucs, label='Val AUC')
plt.legend()
plt.title('AUC per Epoch')

plt.show()

In [ ]:
# Save model if needed
torch.save(model.state_dict(), 'convnext_chexpert.pth')